# Step 1: Import all necessary libraries

In [67]:
# Libraries
import pandas as pd
from traffic.data import aircraft
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


## Step 2: Import existing bottleneck features

In [68]:
helper_bottleneck_features = pd.read_parquet('data/processed/helper_bottleneck_features.parquet')

In [69]:
helper_bottleneck_features.head()

,flight_id,taxi_start,taxi_end,sum_p10_hist_seg_time,sum_median_hist_seg_time,sum_p90_hist_seg_time,sum_seg_length_m,n_intersections_passed,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34
0,GSW444_577,2025-01-01 05:34:37+00:00,2025-01-01 05:37:14+00:00,NaN,NaN,NaN,556.0,3,0,1,0,0,0
1,THY8MQ_937,2025-01-01 05:43:08+00:00,2025-01-01 06:13:11+00:00,NaN,NaN,NaN,1301.9,7,1,0,0,0,0
2,EDW402_523,2025-01-01 05:56:41+00:00,2025-01-01 06:19:41+00:00,NaN,NaN,NaN,1850.2,12,1,0,0,0,0
3,SWR8YH_482,2025-01-01 06:02:22+00:00,2025-01-01 06:23:16+00:00,NaN,NaN,NaN,2322.2,13,1,0,0,0,0
4,AFR94BH_056,2025-01-01 06:04:22+00:00,2025-01-01 06:15:57+00:00,NaN,NaN,NaN,1087.0,6,1,0,0,0,0


## Step 3: Import processed trajectories

In [70]:
trajs_processed = pd.read_parquet('data/processed/trajs_processed.parquet')

In [71]:
trajs_processed.head()

,timestamp,icao24,latitude,longitude,groundspeed,track,vertical_rate,callsign,onground,alert,spi,squawk,altitude,geoaltitude,lastcontact,serials,hour,flight_id,registration,typecode,track_unwrapped,cumdist,compute_gs,compute_track,cut_runway,x,y,groundspeed_xy,track_xy,latitude_kalman,longitude_kalman,groundspeed_kalman,track_kalman
0,2025-01-01 10:39:44+00:00,34745a,47.455069,8.561117,<NA>,NaN,<NA>,AEA85MU,True,False,False,3073,<NA>,<NA>,1735727983.932,[1996109518],2025-01-01 10:00:00+00:00,AEA85MU_045,None,None,NaN,NaN,9.282293,61.693495,28,984.881396,-332.010239,9.282311,61.683913,47.455071,8.561416,4.564451,21.318971
1,2025-01-01 10:39:45+00:00,34745a,47.455089,8.561172,<NA>,NaN,<NA>,AEA85MU,True,False,False,3073,<NA>,<NA>,1735727983.932,[1996109518],2025-01-01 10:00:00+00:00,AEA85MU_045,None,None,NaN,NaN,9.282293,61.693495,28,989.085247,-329.745178,9.282310,61.683887,47.455089,8.561426,4.665585,19.967358
2,2025-01-01 10:39:46+00:00,34745a,47.45511,8.561228,<NA>,NaN,<NA>,AEA85MU,True,False,False,3073,<NA>,<NA>,1735727983.932,[1996109518],2025-01-01 10:00:00+00:00,AEA85MU_045,None,None,NaN,NaN,9.282293,61.693495,28,993.289094,-327.480114,9.282308,61.683862,47.455109,8.561441,4.752577,18.311624
3,2025-01-01 10:39:47+00:00,34745a,47.45513,8.561284,<NA>,NaN,<NA>,AEA85MU,True,False,False,3073,<NA>,<NA>,1735727983.932,[1996109518],2025-01-01 10:00:00+00:00,AEA85MU_045,None,None,NaN,NaN,9.282293,61.693495,28,997.492938,-325.215047,9.282306,61.683812,47.455131,8.561450,4.902902,15.550597
4,2025-01-01 10:39:48+00:00,34745a,47.45515,8.56134,<NA>,NaN,<NA>,AEA85MU,True,False,False,3073,<NA>,<NA>,1735727983.932,[1996109518],2025-01-01 10:00:00+00:00,AEA85MU_045,None,None,NaN,NaN,9.282293,61.693495,28,1001.696779,-322.949977,9.282303,61.683761,47.455154,8.561456,5.030594,13.485953


# Step 4: Import processed meteorological data

In [72]:
meteo_processed = pd.read_parquet('data/processed/meteo_processed.parquet')

## Step 5: Feature Engineering

In the following steps, additional features based on the processed trajectories are being created and then merged with the existing bottleneck features.

### Step 5.1: Operational Features

First, we create a features dataframe and define the target variable (taxi out time) for each flight_id.

In [73]:
features_df = helper_bottleneck_features[
    ["flight_id", "taxi_start", "taxi_end"]
].copy()

features_df["taxi_time_s"] = (
    features_df["taxi_end"] - features_df["taxi_start"]
).dt.total_seconds()

Before defining a filtering threshold, the distribution of taxi-out times is inspected using summary statistics and lower quantiles to understand the realistic range of values.

In [74]:
def inspect_taxi_time_distribution(features_df):

    print("Summary statistics:")
    print(features_df["taxi_time_s"].describe())

    print("\nLower quantiles:")
    print(
        features_df["taxi_time_s"].quantile(
            [0.01, 0.02, 0.05, 0.10]
        )
    )


inspect_taxi_time_distribution(features_df)

Summary statistics:
count     287.000000
mean      644.515679
std       432.661885
min         3.000000
25%       365.500000
50%       583.000000
75%       785.500000
max      2378.000000
Name: taxi_time_s, dtype: float64

Lower quantiles:
0.01     10.44
0.02     48.16
0.05    107.30
0.10    135.80
Name: taxi_time_s, dtype: float64


A configurable minimum taxi-out time parameter is defined to allow flexible adjustment of the filtering threshold as the dataset grows or data characteristics change.


 <font color='red'> CHANGE THE PARAMETER IN THE CELL BELOW: </font>

In [75]:
MIN_TAXI_TIME_S = 60

Flights with taxi-out times below the defined minimum threshold are excluded from further analysis to remove incomplete or operationally implausible trajectories.

In [76]:
features_df = features_df[
    features_df["taxi_time_s"] >= MIN_TAXI_TIME_S
]

Adding aircraft typecode as a feature and setting datatype as "category"

In [77]:
aircraft_lookup = (
    trajs_processed[["icao24"]]
    .drop_duplicates()
    .assign(
        typecode=lambda df: df["icao24"].apply(
            lambda x: aircraft.get(x).typecode
            if aircraft.get(x) is not None
            else None
        )
    )
)

flight_aircraft = (
    trajs_processed[["flight_id", "icao24"]]
    .drop_duplicates()
    .merge(aircraft_lookup, on="icao24", how="left")
)

features_df = features_df.merge(
    flight_aircraft[["flight_id", "typecode"]],
    on="flight_id",
    how="left"
)

features_df["typecode"] = features_df["typecode"].astype("category")

The queue size feature is calculated within a rolling one-hour lookback window. For each flight, only aircraft that started taxiing within the previous hour and were still taxiing at the current flight's taxi_start time are counted. This keeps the feature operationally realistic, prediction-safe, and computationally scalable for larger datasets.

In [78]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]

queue_size_last_hour = []

for i, t in enumerate(starts):
    n_active = (
        (starts >= t - WINDOW)
        & (starts < t)
        & (ends > t)
    ).sum()

    queue_size_last_hour.append(n_active)

features_df["queue_size"] = queue_size_last_hour

Implementing the average taxi out time of the past 6 hours as a feature

In [79]:
WINDOW = pd.Timedelta(hours=6)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_6h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_6h.append(avg)

features_df["avg_taxi_out_time_6h"] = avg_taxi_out_6h

Implementing the average taxi out time of the past hour as a feature.

In [80]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_1h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_1h.append(avg)

features_df["avg_taxi_out_time_1h"] = avg_taxi_out_1h

Implementing the average taxi out time of the past 15 minutes as feature.

In [81]:
WINDOW = pd.Timedelta(hours=0.25)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_15min = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_15min.append(avg)

features_df["avg_taxi_out_time_15min"] = avg_taxi_out_15min

Implementing bottleneck features

In [82]:
cols_to_merge = [
    "flight_id",
    "n_intersections_passed",
    "sum_seg_length_m",
    "sum_p10_hist_seg_time",
    "sum_median_hist_seg_time",
    "sum_p90_hist_seg_time",
    "TO_runway_28",
    "TO_runway_32",
    "TO_runway_16",
    "TO_runway_10",
    "TO_runway_34",
]

features_df = features_df.merge(
    helper_bottleneck_features[cols_to_merge],
    on="flight_id",
    how="left",
)

### Step 5.2 Temporal Features

Defining a function for cyclic encoding

In [83]:
def sin_cos(x,T):
    return np.sin(2 * np.pi * x / T), np.cos(2 * np.pi * x / T)

Splitting up information from the 'taxi_start' column into month, day of week and minute of day

In [84]:
features_df["month"] = features_df["taxi_start"].dt.month

features_df["day_of_week"] = features_df["taxi_start"].dt.dayofweek
# Monday = 0, Sunday = 6

features_df["minute_of_day"] = (
    features_df["taxi_start"].dt.hour * 60
    + features_df["taxi_start"].dt.minute
)

Encoding month, day of week and minute of day as sin and cos

In [85]:
features_df["month_sin"], features_df["month_cos"] = sin_cos(
    features_df["month"], 12
)

features_df["day_of_week_sin"], features_df["day_of_week_cos"] = sin_cos(
    features_df["day_of_week"], 7
)

features_df["minute_of_day_sin"], features_df["minute_of_day_cos"] = sin_cos(
    features_df["minute_of_day"], 24 * 60
)

### Step 5.3: Meteorological Features

Implementing temperature and spread as  features

In [86]:
meteo_temp = (
    meteo_processed[
        ["reference_timestamp", "tre200s0", "spread"]
    ]
    .rename(
        columns={
            "tre200s0": "temp_at_taxi_start",
            "spread": "spread_at_taxi_start"
        }
    )
)

features_df = pd.merge_asof(
    features_df.sort_values("taxi_start"),
    meteo_temp.sort_values("reference_timestamp"),
    left_on="taxi_start",
    right_on="reference_timestamp",
    direction="backward"
)

features_df = features_df.drop(columns="reference_timestamp")

In [87]:
features_df.head(6)

,flight_id,taxi_start,taxi_end,taxi_time_s,typecode,queue_size,avg_taxi_out_time_6h,avg_taxi_out_time_1h,avg_taxi_out_time_15min,n_intersections_passed,sum_seg_length_m,sum_p10_hist_seg_time,sum_median_hist_seg_time,sum_p90_hist_seg_time,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34,month,day_of_week,minute_of_day,month_sin,month_cos,day_of_week_sin,day_of_week_cos,minute_of_day_sin,minute_of_day_cos,temp_at_taxi_start,spread_at_taxi_start
0,GSW444_577,2025-01-01 05:34:37+00:00,2025-01-01 05:37:14+00:00,157.0,A320,0,NaN,NaN,NaN,3,556.0,NaN,NaN,NaN,0,1,0,0,0,1,2,334,0.5,0.866025,0.974928,-0.222521,0.993572,0.113203,-2.5,0.1
1,THY8MQ_937,2025-01-01 05:43:08+00:00,2025-01-01 06:13:11+00:00,1803.0,B739,0,157.0,157.0,157.0,7,1301.9,NaN,NaN,NaN,1,0,0,0,0,1,2,343,0.5,0.866025,0.974928,-0.222521,0.997250,0.074108,-2.7,0.0
2,EDW402_523,2025-01-01 05:56:41+00:00,2025-01-01 06:19:41+00:00,1380.0,A320,1,157.0,157.0,NaN,12,1850.2,NaN,NaN,NaN,1,0,0,0,0,1,2,356,0.5,0.866025,0.974928,-0.222521,0.999848,0.017452,-2.6,0.3
3,SWR8YH_482,2025-01-01 06:02:22+00:00,2025-01-01 06:23:16+00:00,1254.0,BCS3,2,157.0,157.0,NaN,13,2322.2,NaN,NaN,NaN,1,0,0,0,0,1,2,362,0.5,0.866025,0.974928,-0.222521,0.999962,-0.008727,-2.9,0.3
4,AFR94BH_056,2025-01-01 06:04:22+00:00,2025-01-01 06:15:57+00:00,695.0,NaN,3,157.0,157.0,NaN,6,1087.0,NaN,NaN,NaN,1,0,0,0,0,1,2,364,0.5,0.866025,0.974928,-0.222521,0.999848,-0.017452,-2.9,0.3
5,SWR464L_1020,2025-01-01 06:05:14+00:00,2025-01-01 06:26:28+00:00,1274.0,BCS3,4,157.0,157.0,NaN,12,1880.0,NaN,NaN,NaN,1,0,0,0,0,1,2,365,0.5,0.866025,0.974928,-0.222521,0.999762,-0.021815,-2.9,0.3


Export features as parquet file

In [88]:
output_path = "data/processed/features.parquet"
 
# Save dataframe to pkl.
features_df.to_parquet(output_path)
 
print(f"File \"features\" saved to: {output_path}")
 

File "features" saved to: data/processed/features.parquet
